In [ ]:
!pip install -q transformers datasets peft accelerate evaluate kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00


In [ ]:
import os
import torch
import numpy as np
import pandas as pd

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
import kagglehub

In [ ]:
path = kagglehub.dataset_download("sbhatti/financial-sentiment-analysis")
print(os.listdir(path))

df = pd.read_csv(os.path.join(path, "data.csv"))

label_map = {"negative": 0, "neutral": 1, "positive": 2}

df = df.dropna()
df["label"] = df["Sentiment"].map(label_map)
df = df.rename(columns={"Sentence": "text"})
df = df[["text", "label"]]

100%|██████████| 276k/276k [00:00<00:00, 25.7MB/s]

Extracting files...
['data.csv']


In [ ]:
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

train_data = dataset["train"]
test_data = dataset["test"]

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_data = train_data.map(tokenize, batched=True)
test_data = test_data.map(tokenize, batched=True)

train_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/4673 [00:00<?, ? examples/s]

Map:   0%|          | 0/1169 [00:00<?, ? examples/s]

In [ ]:
train_df = train_data.to_pandas()

# Split by label
pos_df = train_df[train_df["label"] == 2]
neg_df = train_df[train_df["label"] == 0]
neu_df = train_df[train_df["label"] == 1]

print("Label counts:")
print("Positive:", len(pos_df))
print("Negative:", len(neg_df))
print("Neutral :", len(neu_df))

# Safe sampling (important fix)
def safe_sample(df, n):
    return df.sample(min(len(df), n), random_state=42)

client1_df = safe_sample(pos_df, 800)   # positive-heavy
client2_df = safe_sample(neg_df, 800)   # negative-heavy
client3_df = safe_sample(train_df, 800) # mixed

from datasets import Dataset

client1 = Dataset.from_pandas(client1_df)
client2 = Dataset.from_pandas(client2_df)
client3 = Dataset.from_pandas(client3_df)

clients = [client1, client2, client3]

Label counts:
Positive: 1462
Negative: 675
Neutral : 2536


In [ ]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=preds, references=labels)

Base model (NO fine-tuning)

In [ ]:
#
def get_base_model():
    return AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=3
    )

# LoRA model (fine-tuning)
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"]
)

def get_lora_model():
    base_model = get_base_model()
    return get_peft_model(base_model, lora_config)

In [ ]:
base_model = get_base_model()

trainer = Trainer(
    model=base_model,
    eval_dataset=test_data,
    compute_metrics=compute_metrics
)

base_results = trainer.evaluate()
print("Base Model:", base_results)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Base Model: {'eval_loss': 1.0901979207992554, 'eval_model_preparation_time': 0.0019, 'eval_accuracy': 0.40034217279726264, 'eval_runtime': 4.7102, 'eval_samples_per_second': 248.182, 'eval_steps_per_second': 31.209}


In [ ]:
central_model = get_lora_model()

training_args = TrainingArguments(
    output_dir="./central",
    per_device_train_batch_size=8,
    num_train_epochs=2,
    logging_steps=10,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=central_model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics
)

trainer.train()

central_results = trainer.evaluate()
print("Centralized Model:", central_results)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
10,1.061742
20,1.002710
30,1.005529
40,1.053385
50,0.954659
60,0.977212
70,0.931292
80,0.874070
90,0.911713
100,0.977644


Centralized Model: {'eval_loss': 0.5923883318901062, 'eval_accuracy': 0.7305389221556886, 'eval_runtime': 4.6488, 'eval_samples_per_second': 251.463, 'eval_steps_per_second': 31.621, 'epoch': 2.0}


In [ ]:
def train_client(global_model, dataset, client_id):

    model = get_lora_model()
    model.load_state_dict(global_model.state_dict())

    training_args = TrainingArguments(
        output_dir=f"./client_{client_id}",
        per_device_train_batch_size=8,
        num_train_epochs=1,
        save_strategy="no",
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset
    )

    trainer.train()
    return model

In [ ]:
def federated_average(models):

    global_dict = models[0].state_dict()

    for key in global_dict.keys():
        global_dict[key] = torch.stack(
            [m.state_dict()[key].float() for m in models]
        ).mean(0)

    new_model = get_lora_model()
    new_model.load_state_dict(global_dict)

    return new_model

In [ ]:
global_model = get_lora_model()

rounds = 3

for r in range(rounds):
    print(f"\nROUND {r+1}")

    client_models = []

    for i, client_data in enumerate(clients):
        print(f"Client {i+1} training...")
        m = train_client(global_model, client_data, i)
        client_models.append(m)

    global_model = federated_average(client_models)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



ROUND 1
Client 1 training...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
trainer = Trainer(
    model=global_model,
    eval_dataset=test_data,
    compute_metrics=compute_metrics
)

fed_results = trainer.evaluate()
print("Federated Model:", fed_results)

Federated Model: {'eval_loss': 1.5736708641052246, 'eval_model_preparation_time': 0.0103, 'eval_accuracy': 0.33361847733105215, 'eval_runtime': 5.2384, 'eval_samples_per_second': 223.158, 'eval_steps_per_second': 28.062}


In [ ]:
print("\n====== FINAL COMPARISON ======")
print("Base Model Accuracy:", base_results["eval_accuracy"])
print("Centralized Accuracy:", central_results["eval_accuracy"])
print("Federated Accuracy:", fed_results["eval_accuracy"])


====== FINAL COMPARISON ======
Base Model Accuracy: 0.40034217279726264
Centralized Accuracy: 0.7305389221556886
Federated Accuracy: 0.33361847733105215
